In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import cv2, os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle

In [4]:
DATASET_DIR = '/content/drive/MyDrive/Facial_Data'

class_names = sorted([
    folder for folder in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, folder))
])
IMG_SIZE = 128

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

print("Detected Classes:", class_names)
print("Total Classes:", len(class_names))

Detected Classes: ['21-45902-3', '22-46139-1', '22-46258-1', '22-46275-1', '22-46342-1', '22-46536-1', '22-46590-1', '22-46887-1', '22-46983-1', '22-47813-2', '22-47892-2', '22-47898-2', '22-47968-2', '22-48021-2', '22-48023-2', '22-48091-2', '22-48133-2', '22-48205-2', '22-48569-3', '22-48582-3', '22-48833-3', '22-49037-3', '22-49068-3', '22-49196-3', '22-49338-3', '22-49355-3', '22-49370-3', '22-49421-3', '22-49451-3', '22-49507-3', '22-49538-3', '22-49575-3', '22-49609-3', '22-49643-3', '22-49783-3', '22-49791-3', '22-49800-3', '22-49824-3', '22-49843-3', '22-49852-3', '22-49862-3', '23-50066-1', '23-50158-1', '23-50254-1', '23-50277-1', '23-50346-1', '23-51127-1']
Total Classes: 47


In [5]:
def load_faces(base_dir):
    X, labels = [], []
    for label, person in enumerate(class_names):
        person_dir = os.path.join(base_dir, person)
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)
            if len(faces) == 0:
                continue

            x, y, w, h = faces[0]
            face = img[y:y+h, x:x+w]
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))

            X.append(face)
            labels.append(label)

    return np.array(X), np.array(labels)

X, Y = load_faces(DATASET_DIR)
print("Dataset Shape:", X.shape, Y.shape)

X, Y = shuffle(X, Y, random_state=42)
X = preprocess_input(X)


Dataset Shape: (716, 128, 128, 3) (716,)


In [6]:
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True

for layer in base_model.layers[:100]:
    layer.trainable = False

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

inputs = layers.Input(shape=(128, 128, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
NUM_CLASSES = len(class_names)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 47)             │         6,063 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,428,015 (9.26 MB)

 Trainable params: 2,031,471 (7.75 MB)

 Non-trainable params: 396,544 (1.51 MB)

In [7]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)
model.fit(X, Y, epochs=30, batch_size=32, validation_split=0.2, callbacks=[early_stop, checkpoint])
model.save("/content/drive/MyDrive/DeepAttend.keras")

with open("/content/drive/MyDrive/class_names.json", "w") as f:
    json.dump(class_names, f)


Epoch 1/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 21s 235ms/step - accuracy: 0.2255 - loss: 3.2326 - val_accuracy: 0.0417 - val_loss: 3.9364
Epoch 2/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 105ms/step - accuracy: 0.5577 - loss: 1.7465 - val_accuracy: 0.0625 - val_loss: 5.5659
Epoch 3/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - accuracy: 0.6993 - loss: 1.0742 - val_accuracy: 0.0764 - val_loss: 6.4042
Epoch 4/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.7780 - loss: 0.8837 - val_accuracy: 0.0417 - val_loss: 6.8088
Epoch 5/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - accuracy: 0.8514 - loss: 0.6265 - val_accuracy: 0.0903 - val_loss: 6.4370
Epoch 6/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.8706 - loss: 0.4886 - val_accuracy: 0.1736 - val_loss: 5.9016
